In [ ]:
# Cell 1 - Imports
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.spatial import cKDTree
from scipy.interpolate import griddata, RegularGridInterpolator
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
# Cell 2 - Configuration
# ----------------------------------------------------------------------
# Paths & files
# ----------------------------------------------------------------------
DATA_wave = Path('/export/lv9/projects/dws/model_output/archived_runs/effective_fetch/spinup_01/')
DATA_orig = Path('/export/lv9/projects/dws/model_output/archived_runs/effective_fetch/spinup_01/')

FILE_PATTERN_wave = 'yearly_dws_500m.3d.2015_effective_fetch.nc'
FILE_PATTERN_orig = 'yearly_dws_500m.3d.2015_effective_fetch.nc'

Validation_DATA_DIR = Path('/export/lv9/projects/dws/results/validation/pelagic/')
MWTL_FILE = Validation_DATA_DIR / 'Field' / 'MWTL_Turbidity.csv'

# TrilaWatt (adjust if needed)
TrilaWatt_data_dir = Validation_DATA_DIR / 'TrilaWatt' / 'Hydrodynamik' / '2015'
TW_DAILY_NC = TrilaWatt_data_dir / 'trilawatt_daily_2015.nc'

# ----------------------------------------------------------------------
# Model settings
# ----------------------------------------------------------------------
SURFACE_LAYER_INDEX = 10          # 0-based; top layer in your convention
MODEL_SURFACE_LAYER_INDEX = 10
USE_DAILY_MEAN = False
ROLLING_WINDOW = None             # e.g. 3 or None

# Domain subset (Python slices)
X_SLICE = (1, 320)
Y_SLICE = (1, 190)

# Variables of interest
CHLA_VAR = 'Chla'
ELEV_VAR = 'elev'
Bathymetry_VAR = 'bathymetry'

# Unit conversion for TrilaWatt SSC (adjust if necessary)
TW_UNIT_SCALE = {
    'suspended_sediment_concentration_2d': 1e6,   # example – verify!
}

TW_SSC_VAR = 'suspended_sediment_concentration_2d'

In [ ]:
# Cell 3 - Helper functions
def find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')


def find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None


def drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da


def pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None


def find_bathy_name(ds: xr.Dataset, preferred: str = 'bathymetry') -> str | None:
    if preferred in ds.variables:
        return preferred
    candidates = ('bathymetry', 'depth', 'h', 'H', 'bathy', 'bat', 'topo',
                  'd', 'water_depth', 'waterdepth', 'DEP')
    for name in candidates:
        if name in ds.variables:
            return name
    # fallback: any variable containing 'depth' or 'bathy'
    for v in ds.data_vars:
        if 'depth' in v.lower() or 'bathy' in v.lower():
            return v
    return None


def to_float(values) -> np.ndarray:
    if np.ma.isMaskedArray(values):
        values = np.ma.filled(values, np.nan)
    return np.asarray(values, dtype=float)


def maybe_smooth(series: xr.DataArray, time_dim: str) -> xr.DataArray:
    out = series
    if USE_DAILY_MEAN:
        out = out.resample({time_dim: '1D'}).mean(skipna=True)
    if ROLLING_WINDOW is not None:
        if ROLLING_WINDOW < 1:
            raise ValueError('ROLLING_WINDOW must be >= 1 or None')
        out = out.rolling({time_dim: ROLLING_WINDOW}, center=True).mean()
    return out


def parse_geom(geom_str: str) -> tuple[float, float]:
    """Parse WKT POINT(lon lat) → (lon, lat)."""
    if isinstance(geom_str, str) and geom_str.startswith('POINT'):
        coords_str = geom_str[6:].strip().strip('()')
        lon, lat = map(float, coords_str.split())
        return lon, lat
    raise ValueError(f'Unexpected geom format: {geom_str}')


In [ ]:
# Cell 4 - Load model data
wave_files = sorted(DATA_wave.glob(FILE_PATTERN_wave))
orig_files = sorted(DATA_orig.glob(FILE_PATTERN_orig))

if not wave_files:
    raise FileNotFoundError(f'No files matching {FILE_PATTERN_wave} in {DATA_wave}')
if not orig_files:
    raise FileNotFoundError(f'No files matching {FILE_PATTERN_orig} in {DATA_orig}')

ds_wave = xr.open_mfdataset(
    wave_files,
    combine='nested',
    concat_dim='time',
    decode_times=True,
    data_vars='minimal',
    coords='minimal',
    compat='override',
    join='override',
)

ds_orig = xr.open_mfdataset(
    orig_files,
    combine='nested',
    concat_dim='time',
    decode_times=True,
    data_vars='minimal',
    coords='minimal',
    compat='override',
    join='override',
)

# Use the original SPM dataset as the main working dataset for later validation cells.
ds_2015 = ds_orig

print(f'Loaded {len(wave_files)} wave file(s); variables: {sorted(ds_wave.data_vars)}')
print(f'Loaded {len(orig_files)} SPM file(s); variables: {sorted(ds_orig.data_vars)}')
print(f'Using ds_2015 from DATA_orig with {len(ds_2015.data_vars)} variables')


In [ ]:
# Updated Cell 5 – Load MWTL (keep hourly points)
def load_mwtl(fpath: Path, year: int = 2015):
    """
    Returns
    -------
    mwtl_hourly : original timestamps + SPM (mg/m³) for the requested year
    site_xy     : DataFrame with lon/lat indexed by locatie.code
    """
    raw = pd.read_csv(fpath)
    raw['tijdstip'] = pd.to_datetime(raw['tijdstip'], utc=True).dt.tz_convert(None)

    # Convert units
    raw['SPM_mg_m3'] = raw['numeriekewaarde'] * 1000   # mg/L → mg/m³

    # Site coordinates
    site_geom = (
        raw[['locatie.code', 'geom']]
        .drop_duplicates('locatie.code')
        .set_index('locatie.code')
    )
    site_xy = pd.DataFrame(
        [parse_geom(g) for g in site_geom['geom']],
        index=site_geom.index,
        columns=['lon', 'lat']
    )
    raw = raw.join(site_xy, on='locatie.code')

    # Keep only the requested year (hourly / original timestamps)
    mwtl_hourly = raw[raw['tijdstip'].dt.year == year].copy()

    print(f'Sites: {sorted(mwtl_hourly["locatie.code"].unique())}')
    print(f'Hourly / point records in {year}: {len(mwtl_hourly)}')
    return mwtl_hourly, site_xy


mwtl_2015, site_xy = load_mwtl(MWTL_FILE, year=2015)
print('\nSite coordinates (first 5):')
print(site_xy.head())

In [ ]:
# Cell 6 - Prepare depth-averaged ESS + water depth
def prepare_ess(ds: xr.Dataset) -> tuple[xr.DataArray, str, str, str]:
    """Return depth-averaged ESS, time_dim, y_dim, x_dim."""
    if 'ESS' not in ds.variables:
        raise KeyError(f"'ESS' not found. Available: {sorted(ds.data_vars)}")

    ess = ds['ESS'].squeeze(drop=True)
    td = find_time_dim(ess)
    zd = find_vertical_dim(ess, td)
    ess = drop_duplicate_time(ess, td)

    if zd is not None:
        ess = ess.mean(dim=zd, skipna=True)
    ess = ess.where(ess >= 0)

    h_dims = [d for d in ess.dims if d != td]
    if len(h_dims) != 2:
        raise ValueError(f'Expected 2 horizontal dims, got {ess.dims}')
    y_dim, x_dim = h_dims
    return ess, td, y_dim, x_dim


def prepare_depth(ds: xr.Dataset, time_dim: str) -> xr.DataArray | None:
    name = find_bathy_name(ds)
    if name is None:
        print('WARNING: No water-depth variable found')
        return None
    depth = ds[name].squeeze(drop=True)
    if time_dim in depth.dims:
        depth = depth.mean(dim=time_dim, skipna=True)
    print(f"Using water-depth variable: '{name}'")
    return depth


ess, td, y_dim, x_dim = prepare_ess(ds_2015)
depth = prepare_depth(ds_2015, td)

In [ ]:
# Cell 7 - Build KDTree on model grid
lon_name = pick_coord_name(ds_2015, ('lonc', 'lon', 'longitude'))
lat_name = pick_coord_name(ds_2015, ('latc', 'lat', 'latitude'))
if lon_name is None or lat_name is None:
    raise ValueError('Model has no recognisable lon/lat coordinates')

lon2d = ds_2015[lon_name].values
lat2d = ds_2015[lat_name].values
lon_flat = lon2d.ravel()
lat_flat = lat2d.ravel()
valid_mask = np.isfinite(lon_flat) & np.isfinite(lat_flat)
valid_idx = np.where(valid_mask)[0]

tree = cKDTree(np.column_stack([lon_flat[valid_mask], lat_flat[valid_mask]]))
print(f'KDTree built on {valid_mask.sum()} valid grid points')

In [ ]:
# ------------------------------------------------------------------
# 8 - Time-series comparison (EMOaaS vs MWTL)
# ------------------------------------------------------------------

# Compare both the original and wave-adjusted EMOaaS SPM estimates against MWTL observations.
sites = [s for s in sorted(mwtl_2015['locatie.code'].unique()) if s in site_xy.index]
n_sites = len(sites)

fig, axes = plt.subplots(n_sites, 1, figsize=(13, 4 * n_sites), constrained_layout=True)
axes = np.atleast_1d(axes)

model_sources = [
    ('EMOaaS original', ds_2015, 'tab:blue'),
    ('EMOaaS wave-adjusted', ds_wave, 'tab:green'),
]

for ax, site in zip(axes, sites):
    slon = site_xy.loc[site, 'lon']
    slat = site_xy.loc[site, 'lat']
    depth_vals = []
    model_series = []

    for label, ds_model, color in model_sources:
        if 'ESS' not in ds_model.data_vars:
            print(f'Skipping {label}: ESS not present in dataset')
            continue

        ess_i, td_i, y_dim_i, x_dim_i = prepare_ess(ds_model)
        lon_name_i = pick_coord_name(ds_model, ('lonc', 'lon', 'longitude'))
        lat_name_i = pick_coord_name(ds_model, ('latc', 'lat', 'latitude'))
        if lon_name_i is None or lat_name_i is None:
            raise ValueError(f'{label} model has no recognisable lon/lat coordinates')

        lon2d_i = ds_model[lon_name_i].values
        lat2d_i = ds_model[lat_name_i].values
        lon_flat = lon2d_i.ravel()
        lat_flat = lat2d_i.ravel()
        valid_mask = np.isfinite(lon_flat) & np.isfinite(lat_flat)
        valid_idx_i = np.where(valid_mask)[0]

        tree_i = cKDTree(np.column_stack([lon_flat[valid_mask], lat_flat[valid_mask]]))
        _, nn = tree_i.query([[slon, slat]], k=4)
        fis = valid_idx_i[nn[0]]

        emo_ts_list = []
        depth_i = prepare_depth(ds_model, td_i)

        for fi in fis:
            iy, ix = np.unravel_index(fi, lon2d_i.shape)
            ts = ess_i.isel({y_dim_i: int(iy), x_dim_i: int(ix)})
            emo_ts_list.append(ts)

            if depth_i is not None:
                try:
                    dval = float(depth_i.isel({y_dim_i: int(iy), x_dim_i: int(ix)}).values)
                    if np.isfinite(dval):
                        depth_vals.append(dval)
                except Exception:
                    pass

        emo_ts = sum(emo_ts_list) / len(emo_ts_list)
        emo_times = pd.to_datetime(emo_ts[td_i].values)
        emo_vals = emo_ts.values
        mask_2015 = pd.DatetimeIndex(emo_times).year == 2015

        ax.plot(
            emo_times[mask_2015], emo_vals[mask_2015],
            lw=1.0, color=color, alpha=0.75,
            label=f'{label} ESS (depth-avg, nearest 4)'
        )
        model_series.append(emo_vals[mask_2015])

    depth_str = f", depth ≈ {np.mean(depth_vals):.1f} m" if depth_vals else ""

    # ---- MWTL observations ----
    obs_site = mwtl_2015[mwtl_2015['locatie.code'] == site]
    ax.scatter(obs_site['tijdstip'], obs_site['SPM_mg_m3'],
               s=22, color='tab:orange', alpha=0.85, linewidths=0,
               label='MWTL obs', zorder=3)

    # ---- Axis limits ----
    # Use the full EMOaaS model time span as x-axis limits so the plot follows the modeled series.
    emo_plot_times = []
    for label, ds_model, color in model_sources:
        if 'ESS' not in ds_model.data_vars:
            continue
        ess_i, td_i, y_dim_i, x_dim_i = prepare_ess(ds_model)
        lon_name_i = pick_coord_name(ds_model, ('lonc', 'lon', 'longitude'))
        lat_name_i = pick_coord_name(ds_model, ('latc', 'lat', 'latitude'))
        if lon_name_i is None or lat_name_i is None:
            continue

        lon2d_i = ds_model[lon_name_i].values
        lat2d_i = ds_model[lat_name_i].values
        lon_flat = lon2d_i.ravel()
        lat_flat = lat2d_i.ravel()
        valid_mask = np.isfinite(lon_flat) & np.isfinite(lat_flat)
        valid_idx_i = np.where(valid_mask)[0]

        tree_i = cKDTree(np.column_stack([lon_flat[valid_mask], lat_flat[valid_mask]]))
        _, nn = tree_i.query([[slon, slat]], k=4)
        fis = valid_idx_i[nn[0]]

        emo_ts_list = []
        for fi in fis:
            iy, ix = np.unravel_index(fi, lon2d_i.shape)
            emo_ts_list.append(ess_i.isel({y_dim_i: int(iy), x_dim_i: int(ix)}))

        if emo_ts_list:
            emo_ts = sum(emo_ts_list) / len(emo_ts_list)
            emo_times = pd.to_datetime(emo_ts[td_i].values)
            emo_plot_times.extend(pd.DatetimeIndex(emo_times)[pd.DatetimeIndex(emo_times).year == 2015].tolist())

    if emo_plot_times:
        ax.set_xlim(min(emo_plot_times), max(emo_plot_times))
    else:
        ax.set_xlim(pd.Timestamp('2015-01-01'), pd.Timestamp('2016-01-01'))

    obs_vals = obs_site['SPM_mg_m3'].values
    finite_obs = obs_vals[np.isfinite(obs_vals)]
    if finite_obs.size:
        ymin = min(0.0, float(finite_obs.min()))
        ymax = float(finite_obs.max())
        ypad = 0.05 * max(ymax - ymin, 1.0)
        ax.set_ylim(ymin - ypad, ymax + ypad)

    units = ds_2015['ESS'].attrs.get('units', 'mg/m³')
    ax.set_ylabel(f'SPM [{units}]')
    ax.set_title(f"{site}  (lon={slon:.3f}°, lat={slat:.3f}°{depth_str})")
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper right')
    ax.set_xlabel('Time (2015)')

fig.suptitle('SPM: EMOaaS original vs wave-adjusted vs MWTL observations (2015)',
             fontsize=13, fontweight='bold')
fig.autofmt_xdate()
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from scipy.spatial import cKDTree

# ---------------------------------------------------------------------------
# Paths & constants
# ---------------------------------------------------------------------------
MWTL_Turbidity_2015 = 'MWTL_Turbidity.csv'
_fpath = Validation_DATA_DIR / 'Field' / MWTL_Turbidity_2015

WAD_SITE_COL = 'selection'
WAD_VALUE_COL = 'value'
MWTL_SITE_COL = 'locatie.code'
MWTL_VALUE_COL = 'numeriekewaarde'
TARGET_SITES = ['DOOVBWT', 'DANTZGT']

wad_emoaas_csv = 'wave_modified.csv'
DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/effective_fetch/spinup_01')
SPM_DIR = Path('/export/lv9/projects/dws/results/validation/SPM/')

# ---------------------------------------------------------------------------
# 1. MWTL observations (common to both models)
# ---------------------------------------------------------------------------
mwtl_raw = pd.read_csv(_fpath)

for _column in (MWTL_SITE_COL, MWTL_VALUE_COL, 'tijdstip'):
    if _column not in mwtl_raw.columns:
        raise KeyError(
            f"MWTL CSV column '{_column}' not found. "
            f"Available columns: {list(mwtl_raw.columns)}"
        )

mwtl_raw['tijdstip'] = pd.to_datetime(
    mwtl_raw['tijdstip'], utc=True, errors='coerce'
).dt.tz_convert(None)
mwtl_raw[MWTL_VALUE_COL] = pd.to_numeric(
    mwtl_raw[MWTL_VALUE_COL], errors='coerce'
)
mwtl_raw['date'] = mwtl_raw['tijdstip'].dt.normalize()

mwtl_daily = (
    mwtl_raw.dropna(subset=[MWTL_SITE_COL, 'date', MWTL_VALUE_COL])
    .loc[lambda frame: frame[MWTL_SITE_COL].isin(TARGET_SITES)]
    .groupby([MWTL_SITE_COL, 'date'], as_index=False)[MWTL_VALUE_COL]
    .mean()
    .rename(columns={MWTL_VALUE_COL: 'MWTL_SPM_mg_m3'})
)
mwtl_daily['MWTL_SPM_mg_m3'] *= 1000.0  # mg/L -> mg/m3

mwtl_2015 = (
    mwtl_daily[mwtl_daily['date'].dt.year == 2015]
    .rename(columns={MWTL_SITE_COL: 'site'})
    .copy()
)

# ---------------------------------------------------------------------------
# 2. EMOaaS (adj_Tz) time series
# ---------------------------------------------------------------------------
wad_emoaas_df = pd.read_csv(
    SPM_DIR / wad_emoaas_csv,
    parse_dates=['date'],
    skiprows=10,
)

for _column in (WAD_SITE_COL, WAD_VALUE_COL, 'date'):
    if _column not in wad_emoaas_df.columns:
        raise KeyError(
            f"EMOaaS CSV column '{_column}' not found. "
            f"Available columns: {list(wad_emoaas_df.columns)}"
        )

# check the loaded DataFrame
print(wad_emoaas_df.head())


In [ ]:
import matplotlib.dates as mdates

# ---------------------------------------------------------------------------
# 3. EMOaaS (wad_emoaas_df) selections for the target sites
# ---------------------------------------------------------------------------
# Multiply EMOaaS values by this to get mg/m3 (use 1000.0 if the CSV is in mg/L)
WAD_UNIT_SCALE = 1.0

# Map every selection belonging to a target site to that site, e.g.
# 'MWTL-DANTZGT' / 'DANTZGT-flat-1' -> 'DANTZGT'. The rest of the name
# ('MWTL', 'flat-1', ...) is the selection kind, used to keep colours consistent.
selection_to_site = {}
selection_to_kind = {}
for _selection in wad_emoaas_df[WAD_SITE_COL].dropna().unique():
    for _site in TARGET_SITES:
        if _site in str(_selection):
            selection_to_site[_selection] = _site
            selection_to_kind[_selection] = str(_selection).replace(_site, '').strip('-_ ')

if not selection_to_site:
    raise ValueError(
        f'No EMOaaS selections match {TARGET_SITES}. '
        f'Available: {sorted(wad_emoaas_df[WAD_SITE_COL].dropna().unique())}'
    )
print('EMOaaS selections used:', sorted(selection_to_site))

wad_2015 = wad_emoaas_df[wad_emoaas_df[WAD_SITE_COL].isin(selection_to_site)].copy()
wad_2015['site'] = wad_2015[WAD_SITE_COL].map(selection_to_site)
wad_2015['kind'] = wad_2015[WAD_SITE_COL].map(selection_to_kind)
wad_2015['date'] = (
    pd.to_datetime(wad_2015['date'], utc=True, errors='coerce')
    .dt.tz_convert(None)
    .dt.normalize()
)
wad_2015['EMOaaS_SPM_mg_m3'] = (
    pd.to_numeric(wad_2015[WAD_VALUE_COL], errors='coerce') * WAD_UNIT_SCALE
)
wad_2015 = wad_2015.dropna(subset=['date', 'EMOaaS_SPM_mg_m3'])
wad_2015 = wad_2015[wad_2015['date'].dt.year == 2015]

emo_daily = (
    wad_2015.groupby(['site', 'kind', WAD_SITE_COL, 'date'], as_index=False)['EMOaaS_SPM_mg_m3']
    .mean()
)

# ---------------------------------------------------------------------------
# 4. Pair model and observations on the same day & compute skill metrics
# ---------------------------------------------------------------------------
paired = emo_daily.merge(mwtl_2015, on=['site', 'date'], how='inner')


def spm_skill(frame: pd.DataFrame) -> pd.Series:
    obs = frame['MWTL_SPM_mg_m3'].to_numpy()
    mod = frame['EMOaaS_SPM_mg_m3'].to_numpy()
    diff = mod - obs
    return pd.Series({
        'n': len(frame),
        'obs_mean': obs.mean(),
        'model_mean': mod.mean(),
        'bias': diff.mean(),
        'RMSE': np.sqrt(np.mean(diff ** 2)),
        'r': np.corrcoef(obs, mod)[0, 1] if len(frame) > 2 else np.nan,
    })


if paired.empty:
    print('WARNING: no days where EMOaaS output and MWTL observations overlap in 2015')
else:
    spm_skill_table = (
        paired.groupby(['site', WAD_SITE_COL])[['MWTL_SPM_mg_m3', 'EMOaaS_SPM_mg_m3']]
        .apply(spm_skill)
        .astype({'n': int})
    )
    display(spm_skill_table.round(2))

# ---------------------------------------------------------------------------
# 5. Plot: time series (left) and same-day model vs observation (right)
# ---------------------------------------------------------------------------
# Colours follow the selection kind, so e.g. 'flat-1' has the same colour at both sites
_kind_colors = dict(zip(
    sorted(set(selection_to_kind.values())),
    ['tab:blue', 'tab:orange', 'tab:purple'],
))

fig, axes = plt.subplots(
    len(TARGET_SITES), 2,
    figsize=(15, 4.2 * len(TARGET_SITES)),
    gridspec_kw={'width_ratios': [3, 1]},
    squeeze=False,
    constrained_layout=True,
)

for (_ax_ts, _ax_sc), _site in zip(axes, TARGET_SITES):
    _model_site = emo_daily[emo_daily['site'] == _site]
    _obs_site = mwtl_2015[mwtl_2015['site'] == _site]
    _paired_site = paired[paired['site'] == _site]

    # --- EMOaaS lines + same-day pairs, one per selection ---
    for (_kind, _selection), _subset in _model_site.groupby(['kind', WAD_SITE_COL]):
        _color = _kind_colors.get(_kind, 'tab:gray')
        _ax_ts.plot(
            _subset['date'],
            _subset['EMOaaS_SPM_mg_m3'],
            color=_color,
            lw=1.1,
            label=f'EMOaaS {_selection}'
        )
        _pairs = _paired_site[_paired_site[WAD_SITE_COL] == _selection]
        _ax_sc.scatter(
            _pairs['MWTL_SPM_mg_m3'],
            _pairs['EMOaaS_SPM_mg_m3'],
            color=_color,
            s=30,
            alpha=0.85,
            label=_selection
        )

    # --- MWTL observations ---
    _ax_ts.scatter(
        _obs_site['date'],
        _obs_site['MWTL_SPM_mg_m3'],
        color='black',
        s=28,
        zorder=5,
        label='MWTL observations'
    )

    _ax_ts.set_title(_site)
    _ax_ts.set_ylabel('SPM [mg/m3]')
    _ax_ts.set_xlabel('Date (2015)')
    _ax_ts.set_xlim(pd.Timestamp('2015-01-01'), pd.Timestamp('2016-01-01'))
    _ax_ts.xaxis.set_major_locator(mdates.MonthLocator())
    _ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    _ax_ts.grid(True, alpha=0.25)
    _ax_ts.legend(loc='upper right', fontsize=8)

    # --- Scatter with 1:1 line on equal axes ---
    _vals = _paired_site[['MWTL_SPM_mg_m3', 'EMOaaS_SPM_mg_m3']].to_numpy()
    if _vals.size:
        _lim = (0.0, 1.05 * np.nanmax(_vals))
        _ax_sc.plot(_lim, _lim, color='gray', ls='--', lw=1, label='1:1')
        _ax_sc.set_xlim(_lim)
        _ax_sc.set_ylim(_lim)
        _ax_sc.set_aspect('equal')
        _ax_sc.legend(loc='upper left', fontsize=8)
    _ax_sc.set_title(f'{_site}: same-day pairs')
    _ax_sc.set_xlabel('MWTL SPM [mg/m3]')
    _ax_sc.set_ylabel('EMOaaS SPM [mg/m3]')
    _ax_sc.grid(True, alpha=0.25)

fig.suptitle(
    'EMOaaS vs MWTL SPM observations (2015)',
    fontsize=13,
    fontweight='bold'
)
plt.show()

In [ ]:
import matplotlib.dates as mdates

# ---------------------------------------------------------------------------
# 3. EMOaaS (wad_emoaas_df) selections for the target sites
# ---------------------------------------------------------------------------
# Multiply EMOaaS values by this to get mg/m3 (use 1000.0 if the CSV is in mg/L)
WAD_UNIT_SCALE = 1.0

# Map every selection belonging to a target site to that site, e.g.
# 'MWTL-DANTZGT' / 'DANTZGT-flat-1' -> 'DANTZGT'. The rest of the name
# ('MWTL', 'flat-1', ...) is the selection kind, used to keep colours consistent.
selection_to_site = {}
selection_to_kind = {}
for _selection in wad_emoaas_df[WAD_SITE_COL].dropna().unique():
    for _site in TARGET_SITES:
        if _site in str(_selection):
            selection_to_site[_selection] = _site
            selection_to_kind[_selection] = str(_selection).replace(_site, '').strip('-_ ')

if not selection_to_site:
    raise ValueError(
        f'No EMOaaS selections match {TARGET_SITES}. '
        f'Available: {sorted(wad_emoaas_df[WAD_SITE_COL].dropna().unique())}'
    )
print('EMOaaS selections used:', sorted(selection_to_site))

wad_2015 = wad_emoaas_df[wad_emoaas_df[WAD_SITE_COL].isin(selection_to_site)].copy()
wad_2015['site'] = wad_2015[WAD_SITE_COL].map(selection_to_site)
wad_2015['kind'] = wad_2015[WAD_SITE_COL].map(selection_to_kind)
wad_2015['date'] = (
    pd.to_datetime(wad_2015['date'], utc=True, errors='coerce')
    .dt.tz_convert(None)
    .dt.normalize()
)
wad_2015['EMOaaS_SPM_mg_m3'] = (
    pd.to_numeric(wad_2015[WAD_VALUE_COL], errors='coerce') * WAD_UNIT_SCALE
)
wad_2015 = wad_2015.dropna(subset=['date', 'EMOaaS_SPM_mg_m3'])
wad_2015 = wad_2015[wad_2015['date'].dt.year == 2015]

emo_daily = (
    wad_2015.groupby(['site', 'kind', WAD_SITE_COL, 'date'], as_index=False)['EMOaaS_SPM_mg_m3']
    .mean()
)
emo_daily['month'] = emo_daily['date'].dt.month

# ---------------------------------------------------------------------------
# 4. MWTL monthly climatology from all years with observations
# ---------------------------------------------------------------------------
mwtl_all = mwtl_daily.rename(columns={MWTL_SITE_COL: 'site'}).assign(
    year=lambda frame: frame['date'].dt.year,
    month=lambda frame: frame['date'].dt.month,
)

mwtl_clim = (
    mwtl_all.groupby(['site', 'month'])
    .agg(
        n=('MWTL_SPM_mg_m3', 'size'),
        n_years=('year', 'nunique'),
        median=('MWTL_SPM_mg_m3', 'median'),
        p10=('MWTL_SPM_mg_m3', lambda s: s.quantile(0.10)),
        p25=('MWTL_SPM_mg_m3', lambda s: s.quantile(0.25)),
        p75=('MWTL_SPM_mg_m3', lambda s: s.quantile(0.75)),
        p90=('MWTL_SPM_mg_m3', lambda s: s.quantile(0.90)),
    )
    .reset_index()
)

clim_period = mwtl_all.groupby('site').agg(
    first_year=('year', 'min'),
    last_year=('year', 'max'),
    n_years=('year', 'nunique'),
    n_obs=('MWTL_SPM_mg_m3', 'size'),
)
display(clim_period)

# ---------------------------------------------------------------------------
# 5. Compare EMOaaS 2015 with the climatology
# ---------------------------------------------------------------------------
# Daily EMOaaS values against that month's climatological percentiles
emo_vs_clim = emo_daily.merge(mwtl_clim, on=['site', 'month'], how='inner')
emo_vs_clim['in_p10_p90'] = emo_vs_clim['EMOaaS_SPM_mg_m3'].between(
    emo_vs_clim['p10'], emo_vs_clim['p90']
)

# Monthly medians of EMOaaS vs the climatological monthly median (median vs median,
# since SPM is skewed)
emo_monthly = (
    emo_vs_clim.groupby(['site', 'kind', WAD_SITE_COL, 'month'], as_index=False)
    .agg(
        EMOaaS_median=('EMOaaS_SPM_mg_m3', 'median'),
        clim_median=('median', 'first'),
    )
)


def clim_skill(frame: pd.DataFrame) -> pd.Series:
    clim = frame['clim_median'].to_numpy(dtype=float)
    mod = frame['EMOaaS_median'].to_numpy(dtype=float)
    diff = mod - clim
    return pd.Series({
        'n_months': len(frame),
        'clim_avg': clim.mean(),
        'EMOaaS_avg': mod.mean(),
        'bias': diff.mean(),
        'RMSE': np.sqrt(np.mean(diff ** 2)),
        'r_seasonal': np.corrcoef(clim, mod)[0, 1] if len(frame) > 2 else np.nan,
    })


if emo_monthly.empty:
    print('WARNING: no months where EMOaaS output and MWTL climatology overlap')
else:
    clim_skill_table = (
        emo_monthly.groupby(['site', WAD_SITE_COL])[['clim_median', 'EMOaaS_median']]
        .apply(clim_skill)
        .astype({'n_months': int})
    )
    clim_skill_table['pct_days_in_P10_P90'] = (
        emo_vs_clim.groupby(['site', WAD_SITE_COL])['in_p10_p90'].mean() * 100
    )
    display(clim_skill_table.round(2))

# ---------------------------------------------------------------------------
# 6. Plot: 2015 EMOaaS on the MWTL climatology (left), monthly medians (right)
# ---------------------------------------------------------------------------
# Colours follow the selection kind, so e.g. 'flat-1' has the same colour at both sites
_kind_colors = dict(zip(
    sorted(set(selection_to_kind.values())),
    ['tab:blue', 'tab:orange', 'tab:purple'],
))

# Climatology drawn at mid-month in 2015, padded with Dec/Jan on either side
# so the band covers the whole year
_clim_months = [12] + list(range(1, 13)) + [1]
_clim_x = (
    [pd.Timestamp(2014, 12, 15)]
    + [pd.Timestamp(2015, _m, 15) for _m in range(1, 13)]
    + [pd.Timestamp(2016, 1, 15)]
)

fig, axes = plt.subplots(
    len(TARGET_SITES), 2,
    figsize=(15, 4.2 * len(TARGET_SITES)),
    gridspec_kw={'width_ratios': [3, 1]},
    squeeze=False,
    constrained_layout=True,
)

for (_ax_ts, _ax_sc), _site in zip(axes, TARGET_SITES):
    _model_site = emo_daily[emo_daily['site'] == _site]
    _monthly_site = emo_monthly[emo_monthly['site'] == _site]
    _obs_site = mwtl_2015[mwtl_2015['site'] == _site]
    _clim_site = (
        mwtl_clim[mwtl_clim['site'] == _site]
        .set_index('month')
        .reindex(range(1, 13))
        .loc[_clim_months]
    )

    # --- MWTL climatology band + median ---
    _ax_ts.fill_between(
        _clim_x, _clim_site['p10'], _clim_site['p90'],
        color='0.88', lw=0, label='MWTL clim. P10-P90'
    )
    _ax_ts.fill_between(
        _clim_x, _clim_site['p25'], _clim_site['p75'],
        color='0.74', lw=0, label='MWTL clim. P25-P75'
    )
    _ax_ts.plot(
        _clim_x, _clim_site['median'],
        color='0.3', ls='--', lw=1.4, label='MWTL clim. median'
    )

    # --- EMOaaS 2015 lines + monthly medians, one per selection ---
    for (_kind, _selection), _subset in _model_site.groupby(['kind', WAD_SITE_COL]):
        _color = _kind_colors.get(_kind, 'tab:gray')
        _ax_ts.plot(
            _subset['date'],
            _subset['EMOaaS_SPM_mg_m3'],
            color=_color,
            lw=1.1,
            label=f'EMOaaS {_selection}'
        )
        _months = _monthly_site[_monthly_site[WAD_SITE_COL] == _selection]
        _ax_sc.scatter(
            _months['clim_median'],
            _months['EMOaaS_median'],
            color=_color,
            s=30,
            alpha=0.85,
            label=_selection
        )

    # --- MWTL observations in 2015 ---
    _ax_ts.scatter(
        _obs_site['date'],
        _obs_site['MWTL_SPM_mg_m3'],
        color='black',
        edgecolors='white',
        s=32,
        zorder=5,
        label='MWTL observations 2015'
    )

    if _site in clim_period.index:
        _period = clim_period.loc[_site]
        _clim_label = (
            f"climatology {_period['first_year']}-{_period['last_year']}, "
            f"{_period['n_years']} years, n={_period['n_obs']}"
        )
    else:
        _clim_label = 'no MWTL climatology'
    _ax_ts.set_title(f'{_site} ({_clim_label})')
    _ax_ts.set_ylabel('SPM [mg/m3]')
    _ax_ts.set_xlabel('Date (2015)')
    _ax_ts.set_xlim(pd.Timestamp('2015-01-01'), pd.Timestamp('2016-01-01'))
    _ax_ts.xaxis.set_major_locator(mdates.MonthLocator())
    _ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    _ax_ts.grid(True, alpha=0.25)
    _ax_ts.legend(loc='upper center', fontsize=8, ncol=2)

    # --- Monthly medians: EMOaaS vs climatology, 1:1 line on equal axes ---
    _vals = _monthly_site[['clim_median', 'EMOaaS_median']].to_numpy(dtype=float)
    if _vals.size:
        _lim = (0.0, 1.05 * np.nanmax(_vals))
        _ax_sc.plot(_lim, _lim, color='gray', ls='--', lw=1, label='1:1')
        _ax_sc.set_xlim(_lim)
        _ax_sc.set_ylim(_lim)
        _ax_sc.set_aspect('equal')
        _ax_sc.legend(loc='upper left', fontsize=8)
    _ax_sc.set_title(f'{_site}: monthly medians')
    _ax_sc.set_xlabel('MWTL climatology [mg/m3]')
    _ax_sc.set_ylabel('EMOaaS 2015 [mg/m3]')
    _ax_sc.grid(True, alpha=0.25)

fig.suptitle(
    'EMOaaS 2015 vs MWTL SPM climatology',
    fontsize=13,
    fontweight='bold'
)
plt.show()
